In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.4 Cross-entropy

Cross-entropy is the training loss: the negative log-probability the model
assigned to the correct next token. Lower means the model was more right.

In [ ]:
from torch.nn import functional as F

logits = torch.tensor([[2.0, 0.5, 0.1]])
target = torch.tensor([0])
by_hand = -F.log_softmax(logits, dim=-1)[0, target.item()]
builtin = F.cross_entropy(logits, target)
print("by hand:", by_hand.item(), " F.cross_entropy:", builtin.item())

A useful sanity anchor: a model guessing uniformly over a vocab of size V
scores `ln(V)`. Beating that baseline is the first sign of learning.

In [ ]:
assert torch.allclose(by_hand, builtin)